In [1]:
import os
import sys
import json
import time
import signal
import subprocess
from pathlib import Path

REPO = "https://github.com/eslam-ahmed43/llm-lab.git"
ROOT = Path("/kaggle/working/llm-lab")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
(ROOT / "logs").mkdir(exist_ok=True)
assert (ROOT / "src/train_lora.py").exists(), "train_lora.py is missing: push it to GitHub first"

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

_clock = Path("/tmp/session_start.txt")
if not _clock.exists():
    _clock.write_text(str(time.time()))
SESSION_START = float(_clock.read_text())
MAX_SESSION_H = 10.5


def hours_used():
    return (time.time() - SESSION_START) / 3600


def check_budget(next_step_h):
    if hours_used() + next_step_h > MAX_SESSION_H:
        raise RuntimeError(f"Time budget: {hours_used():.1f}h used, next step needs ~{next_step_h}h")


print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("git log --oneline -1; ls src", shell=True, capture_output=True, text=True).stdout)
print(sys.version)

Cloning into '/kaggle/working/llm-lab'...


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

c4c121a Add LoRA/QLoRA training script (Exp 4)
bench_inference.py
hf_batched_server.py
hf_server.py
quantize.py
train_lora.py

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
%%bash
set -e
pip install -q bitsandbytes datasets accelerate
python -c "
import bitsandbytes, transformers, torch
print('bitsandbytes', bitsandbytes.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)
from datasets import load_dataset
load_dataset('dair-ai/emotion', 'split')
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-1.5B-Instruct')
print('dataset and model cached')
"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 40.5 MB/s eta 0:00:00
bitsandbytes 0.50.2 | transformers 5.16.1 | torch 2.11.0+cu128
dataset and model cached


Fetching 10 files: 100%|██████████| 10/10 [00:07<00:00,  1.36it/s]


In [3]:
def run_train(tag, args, timeout_s=1800):
    out = ROOT / "results" / f"lora_{tag}.json"
    if out.exists():
        print(f"{tag}: already done")
        return out
    check_budget(0.25)
    log_path = ROOT / "logs" / f"lora_{tag}.log"
    cmd = [sys.executable, "-X", "faulthandler", "-u", str(ROOT / "src/train_lora.py"), "run",
           "--out", str(out)] + args
    env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}

    def new_lines(shown):
        lines = log_path.read_text(errors="replace").splitlines()
        for line in lines[shown:]:
            if line.strip() and "Loading weights" not in line and "unauthenticated" not in line:
                print(line[:200], flush=True)
        return len(lines)

    start, shown = time.time(), 0
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(ROOT))
        while proc.poll() is None:
            time.sleep(10)
            shown = new_lines(shown)
            if time.time() - start > timeout_s:
                os.kill(proc.pid, signal.SIGABRT)
                time.sleep(3)
                proc.kill()
                print(log_path.read_text(errors="replace")[-3000:])
                raise TimeoutError(f"{tag} exceeded {timeout_s}s")
    new_lines(shown)
    if proc.returncode != 0:
        print(log_path.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{tag} failed with exit code {proc.returncode}")
    return out

In [4]:
SMOKE = ["--steps", "40", "--eval-n", "200", "--train-limit", "2000"]
run_train("smoke_zeroshot", ["--mode", "zeroshot", "--eval-n", "200"])
run_train("smoke_lora", ["--mode", "lora", "--rank", "8", "--targets", "all", "--seed", "0"] + SMOKE)
run_train("smoke_qlora", ["--mode", "qlora", "--rank", "8", "--targets", "all", "--seed", "0"] + SMOKE)
for tag in ["smoke_zeroshot", "smoke_lora", "smoke_qlora"]:
    r = json.loads((ROOT / "results" / f"lora_{tag}.json").read_text())
    print(tag, "| trainable:", r.get("trainable_params"), "| eval:", r["eval"],
          "| train:", {k: round(v, 3) for k, v in r.get("train", {}).items() if isinstance(v, float)})
for f in (ROOT / "results").glob("lora_smoke_*"):
    f.unlink()

[zeroshot] loading Qwen/Qwen2.5-1.5B-Instruct
[zeroshot] accuracy 0.5250, macro-F1 0.4281
[zeroshot] saved /kaggle/working/llm-lab/results/lora_smoke_zeroshot.json
[lora_r8_all_s0] loading Qwen/Qwen2.5-1.5B-Instruct
[lora_r8_all_s0] 196 adapted modules, 9,232,384 trainable params
/kaggle/working/llm-lab/src/train_lora.py:249: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `o
  sched.step()
[lora_r8_all_s0] accuracy 0.7100, macro-F1 0.6279
[lora_r8_all_s0] saved /kaggle/working/llm-lab/results/lora_smoke_lora.json
[qlora_r8_all_s0] loading Qwen/Qwen2.5-1.5B-Instruct
[qlora_r8_all_s0] 196 adapted modules, 9,232,384 trainable params
[qlora_r8_all_s0] accuracy 0.6850, macro-F1 0.5881
[qlora_r8_all_s0] saved /kaggle/working/llm-lab/results/lora_smoke_qlora.json
smoke_zeroshot | trainable: 0 | eval: {'accuracy': 0.525, 'macro_f1': 0.4280958695718031, 'n': 200} | train: {}
smoke_lora | trai

In [5]:
import traceback

COMMON = ["--steps", "600"]
runs = [("zeroshot", ["--mode", "zeroshot"])]
for seed in [0, 1, 2]:
    for targets in ["attn", "all"]:
        for rank in [4, 8, 16, 32]:
            runs.append((f"lora_r{rank}_{targets}_s{seed}",
                         ["--mode", "lora", "--rank", str(rank), "--targets", targets, "--seed", str(seed)] + COMMON))
    for rank in [8, 16]:
        runs.append((f"qlora_r{rank}_all_s{seed}",
                     ["--mode", "qlora", "--rank", str(rank), "--targets", "all", "--seed", str(seed)] + COMMON))

for tag, args in runs:
    try:
        run_train(tag, args)
    except Exception:
        traceback.print_exc()

[zeroshot] loading Qwen/Qwen2.5-1.5B-Instruct
[zeroshot] accuracy 0.5120, macro-F1 0.4001
[zeroshot] saved /kaggle/working/llm-lab/results/lora_zeroshot.json
[lora_r4_attn_s0] loading Qwen/Qwen2.5-1.5B-Instruct
[lora_r4_attn_s0] 112 adapted modules, 1,089,536 trainable params
/kaggle/working/llm-lab/src/train_lora.py:249: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `o
  sched.step()
  step 100/600 loss 1.3496
  step 200/600 loss 0.6792
  step 300/600 loss 0.5347
  step 400/600 loss 0.4372
  step 500/600 loss 0.3341
  step 600/600 loss 0.2853
[lora_r4_attn_s0] accuracy 0.8995, macro-F1 0.8427
[lora_r4_attn_s0] saved /kaggle/working/llm-lab/results/lora_lora_r4_attn_s0.json
[lora_r8_attn_s0] loading Qwen/Qwen2.5-1.5B-Instruct
[lora_r8_attn_s0] 112 adapted modules, 2,179,072 trainable params
/kaggle/working/llm-lab/src/train_lora.py:249: UserWarning: Detected call of `lr_scheduler.st

In [6]:
import shutil

subprocess.run([sys.executable, str(ROOT / "src/train_lora.py"), "summarize",
                "--results-dir", str(ROOT / "results"), "--out", str(ROOT / "results/lora_summary.csv")],
               check=True)
shutil.make_archive("/kaggle/working/llm-lab-exp4", "zip", ROOT / "results")
print("Zipped to /kaggle/working/llm-lab-exp4.zip")

    mode targets  rank  runs  trainable_params  acc_mean  acc_std  f1_mean  f1_std  peak_mem_gib  train_seconds  tokens_per_s  final_loss
    lora     all     4     3           4616192    0.9217   0.0010   0.8735  0.0035        9.0676       396.5104     1208.8080      0.1874
    lora     all     8     3           9232384    0.9280   0.0028   0.8783  0.0032        9.1475       394.9656     1213.5512      0.1624
    lora     all    16     3          18464768    0.9270   0.0015   0.8764  0.0055        9.3072       397.5716     1205.5772      0.1533
    lora     all    32     3          36929536    0.9285   0.0035   0.8745  0.0078        9.6622       404.5502     1184.8104      0.1341
    lora    attn     4     3           1089536    0.8933   0.0063   0.8367  0.0053        7.3569       316.2908     1515.4251      0.3280
    lora    attn     8     3           2179072    0.9118   0.0016   0.8569  0.0027        7.3725       316.1329     1516.1689      0.2406
    lora    attn    16     3      